In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

In [2]:
df_p2p = pd.read_csv('../data/raw/final_p2p_log.csv')
df_trans = pd.read_csv('../data/raw/final_trans_log.csv')

df_p2p['EventTime'] = pd.to_datetime(df_p2p['EventTime'])
df_trans['EventTime'] = pd.to_datetime(df_trans['EventTime'])

df_p2p = df_p2p.sort_values(by=['UserID', 'EventTime']).reset_index(drop=True)
df_trans = df_trans.sort_values(by=['UserID', 'EventTime']).reset_index(drop=True)

print(f"P2P транзакций (наш таргет): {df_p2p.shape[0]}")
print(f"Обычных транзакций (история): {df_trans.shape[0]}")

P2P транзакций (наш таргет): 2466068
Обычных транзакций (история): 8444107


In [10]:
df_trans['trans_rank'] = df_trans.groupby('UserID').cumcount() + 1

df_trans['time_diff_trans'] = df_trans.groupby('UserID')['EventTime'].diff().dt.total_seconds().fillna(-1)

df_trans['trans_amt_to_mean_3'] = df_trans['Amount'] / (
    df_trans.groupby('UserID')['Amount'].shift(1).rolling(3, min_periods=1).mean().fillna(df_trans['Amount']) + 1e-5
)

if 'MerchantType' in df_trans.columns:
    merch_dummies = pd.get_dummies(df_trans['MerchantType'], prefix='prop')
    for col in merch_dummies.columns:
        df_trans[col] = df_trans.groupby('UserID')[col].shift(1).rolling(5, min_periods=1).mean().fillna(0)

cols_to_keep = ['UserID', 'EventTime', 'time_diff_trans', 'trans_amt_to_mean_3']
cols_to_keep += [col for col in df_trans.columns if col.startswith('prop_')]

trans_features = df_trans[cols_to_keep].sort_values('EventTime').reset_index(drop=True)

In [11]:
df_p2p['time_diff_p2p'] = df_p2p.groupby('UserID')['EventTime'].diff().dt.total_seconds().fillna(-1)

df_p2p['p2p_amt_to_mean_3'] = df_p2p['Amount'] / (
    df_p2p.groupby('UserID')['Amount'].shift(1).rolling(3, min_periods=1).mean().fillna(df_p2p['Amount']) + 1e-5
)

df_p2p['hour'] = df_p2p['EventTime'].dt.hour
df_p2p['day_of_week'] = df_p2p['EventTime'].dt.dayofweek

In [12]:
df_p2p = df_p2p.sort_values('EventTime').reset_index(drop=True)
trans_features = trans_features.sort_values('EventTime').reset_index(drop=True)

df_final = pd.merge_asof(
    left=df_p2p, 
    right=trans_features, 
    on='EventTime', 
    by='UserID', 
    direction='backward',
    tolerance=pd.Timedelta('7d')
)

cols_to_fill = [c for c in trans_features.columns if c not in ['UserID', 'EventTime']]
df_final[cols_to_fill] = df_final[cols_to_fill].fillna(-1)

df_final['is_cold_user'] = (df_final['time_diff_trans'] == -1).astype(float)

outlier_cols = ['Amount', 'p2p_amt_to_mean_3', 'trans_amt_to_mean_3', 'time_diff_p2p', 'time_diff_trans']
for col in outlier_cols:
    if col in df_final.columns:
        upper_limit = df_final[col].quantile(0.99)
        df_final[col] = df_final[col].clip(upper=upper_limit)

print(f"Итоговый размер: {df_final.shape}")

Итоговый размер: (2466068, 28)


In [14]:
split_index = int(len(df_final) * 0.8)

df_train = df_final.iloc[:split_index].copy()
df_test = df_final.iloc[split_index:].copy()

cols_to_drop = ['Currency'] 
df_train = df_train.drop(columns=[col for col in cols_to_drop if col in df_train.columns])
df_test = df_test.drop(columns=[col for col in cols_to_drop if col in df_test.columns])

df_train.to_csv('../data/processed/train_cleaned.csv', index=False)
df_test.to_csv('../data/processed/test_cleaned.csv', index=False)

print(f"Размер Train: {df_train.shape} | Размер Test: {df_test.shape}")
target = 'IsFraud' if 'IsFraud' in df_train.columns else 'isfraud'
print(f"Доля фрода в Train: {df_train[target].mean():.5f}")

Размер Train: (1972854, 27) | Размер Test: (493214, 27)
Доля фрода в Train: 0.00084
